## Running the Server

During development
```bash
python server.py
```
This enables reload=True, so the server restarts automatically when you edit the code.

For production (recommended)
```bash
uvicorn server:app --host 0.0.0.0 --port 8000 --workers 4
```
Replace 4 with the number of CPU cores on your server if appropriate.

In [ ]:
from fastapi import FastAPI
from pydantic import BaseModel
import uvicorn
import socket
import sqlite3
import import_ipynb
from utils import dbms_controller as db

app = FastAPI(
    title="Attendance API",
    version="1.0"
)

# ==========================================================
# Request Model
# ==========================================================

class AttendanceRequest(BaseModel):
    name: str
    user_id: int
    secret_code: str
    device_id: str


# ==========================================================
# Response Model
# ==========================================================

class AttendanceResponse(BaseModel):
    success: bool
    message: str

# Database Object
conn = sqlite3.connect("utils/attendance.db")


# ==========================================================
# PLACEHOLDER FUNCTIONS
# Replace these with your own implementations
# ==========================================================

def verify_device(device_id: str) -> bool:
    """
    Verify whether the attendance machine is registered.
    """
    return True
    # return your_verify_machine_function(device_id)
    # pass


def verify_user_exists(user_id: int, name: str) -> bool:
    """
    Verify that the user exists in the database.
    """
    return True
    # return your_database_lookup(user_id, name)
    # pass


def verify_secret_code(user_id: int, secret_code: str) -> bool:
    if db.verify_user_hash(conn, secret_code):
        return True
    else:
        return False
    # return your_verify_secret_function(user_id, secret_code)
    # pass


def mark_attendance(user_id: int, device_id: str) -> bool:
    if db.mark_presence(conn,user_id,device_id):
        return True
    else:
        return False
    # return your_mark_attendance_function(user_id)
    # pass


# ==========================================================
# API Endpoint
# ==========================================================

@app.post("/attendance", response_model=AttendanceResponse)
async def attendance(data: AttendanceRequest):

    # Verify Device
    if not verify_device(data.device_id):
        return AttendanceResponse(
            success=False,
            message="Invalid Device"
        )

    # Verify User
    if not verify_user_exists(data.user_id, data.name):
        return AttendanceResponse(
            success=False,
            message="User Not Found"
        )

    # Verify Secret Code
    if not verify_secret_code(data.user_id, data.secret_code):
        return AttendanceResponse(
            success=False,
            message="Invalid Security Code"
        )

    # Mark Attendance
    if not mark_attendance(data.user_id, data.device_id):
        return AttendanceResponse(
            success=False,
            message="Attendance Failed"
        )

    return AttendanceResponse(
        success=True,
        message=f"Attendance Marked Successfully for {data.user_id}"
    )


# ==========================================================
# Health Check
# ==========================================================

@app.get("/")
async def home():
    return {
        "status": "online",
        "message": "Attendance Server Running"
    }


# ==========================================================
# Run Server
# ==========================================================

if __name__ == "__main__":
    hostname = socket.gethostname()
    ip = socket.gethostbyname(hostname)

    print(f"Server running at: http://{ip}:8000")

    uvicorn.run(
        "server:app",
        host="0.0.0.0",
        port=8000,
        reload=True
    )